In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import unicodedata
import category_encoders as ce
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.impute import KNNImputer

from sklearn.base import BaseEstimator, TransformerMixin # Para el imputer personalizado

# Variables discretas (valores enteros contables, como categorías codificadas o conteos)
categorical_cols = [
    "trabaja",
    "creditos_a1",
    "superados_a1",
    "posts_foro",
    "uso_biblioteca",
    "eventos",
    "tutorias",
    "numero_fav",
    "meses_matriculado",
    "provincia",
    "bachillerato",
    "modalidad"
]

# Variables continuas (valores numéricos con posibles decimales)
numerical_cols = [
    "horas_moodle",  # Necesita pasarse de string a float
    "horas_trabajo",    # Muchos NaN, si son todos NaN habría que quitarla
    "nota_acceso",
    "nota_s1",      # Podría considerarse continúa
    "satisfaccion",
    "talla_zapato"  # Podría considerarse continua
]

# Variables de texto libre (respuestas abiertas, tipo string)
string_cols = [
    "comentarios",
    "color_fav",
    "id",
    "nombre",
    "residencia_id",
    "grupo_trabajo"
]

# Variables de fechas
date_cols = [
    "nacimiento"
]

pd.set_option('display.max_columns', None)

df_train_raw = pd.read_csv('./csv/uclm_student_train.csv')
df_train_raw

df_test_raw = pd.read_csv('csv/uclm_student_test.csv')
df_test_raw

df_train = df_train_raw.copy()
df_train["horas_moodle"] = df_train["horas_moodle"].str.extract(r'([\d.]+)').astype(float)

df_test = df_test_raw.copy()
df_test["horas_moodle"] = df_test["horas_moodle"].str.extract(r'([\d.]+)').astype(float)

# TODO: Cambiar este método inicial a un imputador, compatible con pipelines
# He supuesto que te refieres a todo, asiq lo dejo comentado por si acaso 
class TextCleanerImputer(BaseEstimator, TransformerMixin):
    def __init__(self, fill_value='Desconocido'):
        self.fill_value = fill_value

    def _remove_accents(self, text):
        if pd.isna(text):
            return text
        return ''.join(
            c for c in unicodedata.normalize('NFKD', str(text))
            if not unicodedata.combining(c)
        )

    def transform(self, X):
        X_transformed = X.copy()
        for col in X_transformed.columns:
            X_transformed[col] = (
                X_transformed[col]
                .fillna(self.fill_value)
                .astype(str)
                .str.lower()
                .apply(self._remove_accents)
                .str.strip()
                .str.replace(r'[\"\.,]', '', regex=True)
                .str.replace(r'\s+', ' ', regex=True)
            )
        return X_transformed

imputer_categorical = TextCleanerImputer()

# TODO: Probar más imputadores. Integrar si es posible con un Pipeline
imputer_numerical = KNNImputer(n_neighbors=2) 

# Entrenamos el imputador SOLO CON TRAIN
imputer_numerical.fit(df_train[numerical_cols])

# TODO: Implementar el uso de pipelines para hacer más sencillo modificar el tratado de datos. Dijo en clase que había que usarlas xd.
from sklearn.pipeline import Pipeline

df_train[categorical_cols] = pd.DataFrame(imputer_categorical.transform(df_train[categorical_cols]))
df_test[categorical_cols] = pd.DataFrame(imputer_categorical.transform(df_test[categorical_cols]))

# Transformamos train y test con el imputador entrenado con train
df_train[numerical_cols] = pd.DataFrame(imputer_numerical.transform(df_train[numerical_cols]), columns=numerical_cols)
df_test[numerical_cols] = pd.DataFrame(imputer_numerical.transform(df_test[numerical_cols]), columns=numerical_cols)

df_train[string_cols] = pd.DataFrame(imputer_categorical.transform(df_train[string_cols]))


In [18]:
temp = ['comentarios', 'abandono']
df_temp = df_train[temp]
df_temp

# 1. Agrupar y agregar los datos
df_resumen = df_temp.groupby('comentarios')['abandono'].agg(
    total_abandonos='sum',
    total_comentarios='count'
).reset_index()

# 2. CALCULAR LA TASA DE ABANDONO
# La Tasa de Abandono se calcula como (total_abandonos / total_comentarios)
df_resumen['tasa_abandono'] = (
    df_resumen['total_abandonos'] / df_resumen['total_comentarios']
)

# 3. Exportar el DataFrame 'df_resumen' con la nueva columna a un archivo CSV
df_resumen.to_csv('resumen_comentarios_abandono_con_tasa.csv', index=False)

print("El archivo 'resumen_comentarios_abandono_con_tasa.csv' ha sido creado exitosamente.")
print("La nueva columna 'tasa_abandono' contiene la probabilidad de abandono para cada comentario.")

El archivo 'resumen_comentarios_abandono_con_tasa.csv' ha sido creado exitosamente.
La nueva columna 'tasa_abandono' contiene la probabilidad de abandono para cada comentario.


In [23]:
# --- 3. INGENIERÍA DE CARACTERÍSTICAS (Target Encoding Analysis) ---

# Columnas para el estudio de Target Encoding
target_encode_cols = ['comentarios', 'provincia', 'bachillerato', 'meses_matriculado']
resumen_list = []

# --- Lógica Específica para la Clasificación de Nombres en 4 Categorías ---

# 1. Preparar la columna 'nombre'
df_train['primer_nombre'] = df_train['nombre'].str.split(' ').str[0]
df_train['longitud_nombre'] = df_train['primer_nombre'].str.len()

# 2. Definir los umbrales usando cuartiles (Q1, Q2, Q3)
q1 = df_train['longitud_nombre'].quantile(0.25)
q2 = df_train['longitud_nombre'].quantile(0.50) # Mediana
q3 = df_train['longitud_nombre'].quantile(0.75)

# 3. Definir los límites y las etiquetas
# Utilizamos bins dinámicos basados en la distribución real de los nombres
bins = [0, q1, q2, q3, df_train['longitud_nombre'].max() + 1]
labels = ['Muy Corto', 'Corto', 'Largo', 'Muy Largo']

# 4. Crear la nueva variable categórica 'longitud_nombre_cat'
df_train['longitud_nombre_cat'] = pd.cut(
    df_train['longitud_nombre'], 
    bins=bins, 
    labels=labels, 
    right=False, # Intervalos abiertos por la derecha
    include_lowest=True
).astype(str) # Convertir a string para evitar problemas de tipos en el groupby

# Añadimos la nueva columna al análisis
target_encode_cols.append('longitud_nombre_cat') 

# -----------------------------------------------------------------


# --- 4. Bucle de Análisis y Generación del Resumen Final ---

for col in target_encode_cols:
    print(f"Procesando variable: {col}...")

    columna_a_agrupar = col
    
    # 1. Agrupar, contar y sumar los abandonos
    df_resumen_temp = df_train.groupby(columna_a_agrupar)[target_col].agg(
        total_abandonos='sum',
        total_registros='count'
    ).reset_index()

    # 2. Calcular la TASA de abandono (0-1) y el PORCENTAJE de abandono (0-100)
    df_resumen_temp['tasa_abandono'] = (
        df_resumen_temp['total_abandonos'] / df_resumen_temp['total_registros']
    )
    df_resumen_temp['porcentaje_abandono'] = df_resumen_temp['tasa_abandono'] * 100 
    
    # 3. Calcular el porcentaje de cada categoría sobre el total de registros
    total_general = len(df_train)
    df_resumen_temp['porcentaje_total_registros'] = (df_resumen_temp['total_registros'] / total_general) * 100

    # 4. Identificar la variable de estudio
    if col == 'longitud_nombre_cat':
        df_resumen_temp['variable_estudio'] = 'longitud_nombre_categorizada'
    else:
        df_resumen_temp['variable_estudio'] = col
    
    # 5. Renombrar y añadir al listado
    df_resumen_temp.rename(columns={columna_a_agrupar: 'categoria'}, inplace=True)
    resumen_list.append(df_resumen_temp)

# Concatenar todos los DataFrames de resumen en uno solo
df_analisis_final = pd.concat(resumen_list, ignore_index=True)

# Reordenar y limpiar columnas finales
df_analisis_final = df_analisis_final[[
    'variable_estudio', 'categoria', 'total_registros', 
    'total_abandonos', 'tasa_abandono', 'porcentaje_abandono', 
    'porcentaje_total_registros'
]]

# --- 5. Exportar el Análisis a CSV ---
df_analisis_final.to_csv('analisis_tasa_abandono_cuatro_cats_final.csv', index=False, float_format='%.4f')

print("\n-----------------------------------------------------------------------")
print("✅ PROCESO COMPLETADO.")
print("El archivo 'analisis_tasa_abandono_cuatro_cats_final.csv' ha sido generado.")
print("Contiene el análisis de las tasas de abandono para las 5 variables,")
print("incluyendo la clasificación de nombres en 4 categorías y el porcentaje (0-100) corregido.")
print("-----------------------------------------------------------------------")

Procesando variable: comentarios...
Procesando variable: provincia...
Procesando variable: bachillerato...
Procesando variable: meses_matriculado...
Procesando variable: longitud_nombre_cat...

-----------------------------------------------------------------------
✅ PROCESO COMPLETADO.
El archivo 'analisis_tasa_abandono_cuatro_cats_final.csv' ha sido generado.
Contiene el análisis de las tasas de abandono para las 5 variables,
incluyendo la clasificación de nombres en 4 categorías y el porcentaje (0-100) corregido.
-----------------------------------------------------------------------


In [ ]:
import pandas as pd
import numpy as np

# Suponemos que df_train ya está cargado, limpio e imputado, incluyendo la columna 'abandono'.

def analizar_tasa_abandono(df: pd.DataFrame, columna_analisis: str) -> pd.DataFrame:
    """
    Calcula el resumen estadístico de abandono (sumas, conteos, tasas) para una columna categórica dada.

    Args:
        df (pd.DataFrame): El DataFrame de entrenamiento limpio y preprocesado.
        columna_analisis (str): El nombre de la columna categórica a analizar 
                                (e.g., 'provincia', 'comentarios').
                                Si se usa 'nombre', aplica la categorización por longitud del primer nombre.

    Returns:
        pd.DataFrame: Un DataFrame con el análisis de la tasa de abandono por categoría.
    """
    target_col = 'abandono'
    df_temp = df.copy()

    # --- Lógica Específica para la Columna 'nombre' ---
    if columna_analisis == 'nombre':
        # 1. Extraer primer nombre
        df_temp['primer_nombre'] = df_temp['nombre'].str.split(' ').str[0]
        # 2. Calcular la longitud
        df_temp['longitud_nombre'] = df_temp['primer_nombre'].str.len()
        
        # 3. Definir umbrales y categorizar (cuartiles)
        q1 = df_temp['longitud_nombre'].quantile(0.25)
        q2 = df_temp['longitud_nombre'].quantile(0.50)
        q3 = df_temp['longitud_nombre'].quantile(0.75)
        
        bins = [0, q1, q2, q3, df_temp['longitud_nombre'].max() + 1]
        labels = ['Muy_Corto', 'Corto', 'Largo', 'Muy_Largo'] # Usamos '_' para el nombre de la columna

        df_temp['longitud_nombre_cat'] = pd.cut(
            df_temp['longitud_nombre'], 
            bins=bins, 
            labels=labels, 
            right=False, 
            include_lowest=True
        ).astype(str)
        
        # Cambiamos la columna a agrupar a la nueva variable categorizada
        columna_a_agrupar = 'longitud_nombre_cat'
        
    else:
        # Para cualquier otra columna, usamos su nombre original
        columna_a_agrupar = columna_analisis

    # --- Análisis y Agregación General ---
    
    # 1. Agrupar, contar y sumar
    df_resumen = df_temp.groupby(columna_a_agrupar)[target_col].agg(
        total_abandonos='sum',
        total_entradas='count'
    ).reset_index()

    # 2. Calcular la Tasa de Abandono
    df_resumen[f'tasa_{columna_analisis}'] = (
        df_resumen['total_abandonos'] / df_resumen['total_entradas']
    )
    
    # 3. Renombrar la columna de la categoría
    df_resumen.rename(columns={columna_a_agrupar: columna_analisis}, inplace=True)

    # 4. Seleccionar y reordenar las columnas de salida
    # Aseguramos que la columna del parámetro esté en primer lugar
    columnas_finales = [columna_analisis, 'total_abandonos', 'total_entradas', f'tasa_{columna_analisis}']
    
    return df_resumen[columnas_finales]





--- Análisis de Provincia ---
  provincia  total_abandonos  total_entradas  tasa_provincia
0     alava               19              83        0.228916
1  albacete             2507            9189        0.272826
2  alicante               25              88        0.284091
3   almeria               22              97        0.226804
4  asturias               15              90        0.166667

--- Análisis de Comentarios ---
                                        comentarios  total_abandonos  \
0           absolutamente implicado en su formacion                0   
1      accede a la plataforma con frecuencia normal                4   
2                    acceso esporadico a biblioteca               10   
3                         actitud correcta en clase               11   
4  actitud desafiante en algunas sesiones practicas               14   

   total_entradas  tasa_comentarios  
0              31          0.000000  
1              34          0.117647  
2              32      

In [8]:
import pandas as pd
import numpy as np

# Asumimos que las funciones 'analizar_tasa_abandono' e 'inferir_sexo' están definidas.
# Y que el DataFrame df_train (limpio y preprocesado) está cargado.

def inferir_sexo(nombre: str) -> str:
    """Infiere el sexo basándose en la terminación del primer nombre (aproximación)."""
    if pd.isna(nombre):
        return 'desconocido'
    primer_nombre = str(nombre).split(' ')[0]
    if primer_nombre.endswith('a'):
        return 'mujer'
    return 'hombre'


def generar_analisis_combinado(df_train: pd.DataFrame) -> pd.DataFrame:
    """
    Realiza análisis de tasa de abandono combinando residencia y grupo de trabajo,
    normalizando los nombres de las categorías combinadas.
    """
    
    # --- 1. PREPARACIÓN DE LAS NUEVAS COLUMNAS ---
    df_temporal = df_train.copy()
    
    # A. Inferencia de Sexo
    df_temporal['sexo_inferido'] = df_temporal['nombre'].apply(inferir_sexo)
    
    # B. CREACIÓN DE LA VARIABLE COMBINADA: residencia_grupo
    # Las columnas ya deberían tener los NaN reemplazados por 'desconocido' y ser str
    
    # 1. Crear la cadena combinada
    df_temporal['residencia_grupo'] = df_temporal['residencia_id'].astype(str) + '---' + df_temporal['grupo_trabajo'].astype(str)
    
    # 2. 🔄 NORMALIZACIÓN DE ETIQUETAS
    # Acortar etiquetas largas de grupo: 'gr00000' -> 'gr0'
    df_temporal['residencia_grupo'] = df_temporal['residencia_grupo'].str.replace(r'gr\d+', lambda m: m.group(0)[:3], regex=True)
    
    # Reemplazar la categoría más ambigua por una clara
    df_temporal['residencia_grupo'] = df_temporal['residencia_grupo'].replace(
        'desconocido---desconocido', 
        'SIN_RESIDENCIA_Y_SIN_GRUPO'
    )
    
    # Reemplazar solo el 'desconocido' para la residencia, cuando el grupo sí existe
    df_temporal['residencia_grupo'] = df_temporal['residencia_grupo'].str.replace(
        'desconocido---', 
        'SIN_RESIDENCIA_EN_GRUPO:'
    )
    
    # C. Lista de variables a analizar (actualizada)
    columnas_a_analizar = [
        'sexo_inferido', 
        'provincia', 
        'comentarios', 
        'nombre', # Para categorización por longitud
        'residencia_grupo' # La nueva variable combinada y normalizada
    ]
    
    resumenes = []
    
    # --- 2. ANÁLISIS MEDIANTE LA FUNCIÓN analizar_tasa_abandono() ---
    for col in columnas_a_analizar:
        df_resumen = analizar_tasa_abandono(df_temporal, col) # La función ya está preparada
        
        # Adaptar la salida para el informe consolidado
        if col == 'nombre':
            df_resumen.rename(columns={'nombre': 'categoria'}, inplace=True)
            df_resumen['variable_estudio'] = 'longitud_nombre_categorizada'
        elif col == 'residencia_grupo':
            df_resumen.rename(columns={col: 'categoria'}, inplace=True)
            df_resumen['variable_estudio'] = 'residencia_grupo_combinado'
        else:
            df_resumen.rename(columns={col: 'categoria'}, inplace=True)
            df_resumen['variable_estudio'] = col

        # Cálculo del porcentaje de abandono para el informe final
        tasa_col_name = f'tasa_{"nombre" if col == "nombre" else col}'
        df_resumen['porcentaje_abandono'] = df_resumen[tasa_col_name] * 100
        df_resumen.drop(columns=[tasa_col_name], inplace=True) 
        
        resumenes.append(df_resumen)

    # --- 3. CONSOLIDACIÓN Y EXPORTACIÓN ---
    df_analisis_consolidado = pd.concat(resumenes, ignore_index=True)
    
    # Reordenar las columnas
    df_analisis_consolidado = df_analisis_consolidado[[
        'variable_estudio', 'categoria', 'total_entradas', 
        'total_abandonos', 'porcentaje_abandono'
    ]]

    # Exportar el análisis a un CSV
    nombre_archivo = 'analisis_consolidado_normalizado.csv'
    df_analisis_consolidado.to_csv(nombre_archivo, index=False, float_format='%.4f')
    
    print(f"\n--- ANÁLISIS CONSOLIDADO Y NORMALIZADO ---")
    print(f"✅ Análisis completado. El archivo '{nombre_archivo}' ha sido generado.")
    print("Las categorías de 'residencia_grupo' han sido normalizadas para mayor claridad.")

    return df_analisis_consolidado

generar_analisis_combinado(df_train=df_train)


--- ANÁLISIS CONSOLIDADO Y NORMALIZADO ---
✅ Análisis completado. El archivo 'analisis_consolidado_normalizado.csv' ha sido generado.
Las categorías de 'residencia_grupo' han sido normalizadas para mayor claridad.


,variable_estudio,categoria,total_entradas,total_abandonos,porcentaje_abandono
0,sexo_inferido,hombre,22858,6384,27.928953
1,sexo_inferido,mujer,17687,4069,23.005597
2,provincia,alava,83,19,22.891566
3,provincia,albacete,9189,2507,27.282621
4,provincia,alicante,88,25,28.409091
...,...,...,...,...,...
1022,residencia_grupo_combinado,e3-p04-h18---gr1,14,5,35.714286
1023,residencia_grupo_combinado,e3-p04-h19---gr0,19,8,42.105263
1024,residencia_grupo_combinado,e3-p04-h19---gr1,18,6,33.333333
1025,residencia_grupo_combinado,e3-p04-h20---gr0,25,7,28.000000
